# Customer Purchase Prediction
**Review 2: Environment Setup · Dataset Exploration · EDA & Reporting**

Team: B. Pemesh (2520030388), N. Madhan (2520030263)

Dataset: `customers.csv` (synthetic sample data, target column `Purchase`: 1 = bought, 0 = did not buy).
Run the cells from top to bottom. In Google Colab, upload `customers.csv` when asked in section 2.

## 1. Environment setup

In [ ]:
import sys, platform
print("Python", sys.version.split()[0], "|", platform.system())
assert sys.version_info >= (3, 9), "Use Python 3.9+"

# Uncomment on a fresh machine / Colab:
# !pip install numpy pandas scikit-learn matplotlib seaborn scipy

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)

import sklearn
for name, mod in [("numpy", np), ("pandas", pd), ("scikit-learn", sklearn), ("seaborn", sns)]:
    print(f"{name:<13}{mod.__version__}")

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
TARGET = "Purchase"

## 2. Load the dataset
Looks for `customers.csv` next to the notebook, then in `data/raw/`. In Colab it asks you to upload the file.

In [ ]:
from pathlib import Path

candidates = ["customers.csv", "data/raw/customers.csv", "../data/raw/customers.csv"]
path = next((p for p in candidates if Path(p).exists()), None)

if path is None:                      # Google Colab fallback
    from google.colab import files
    files.upload()
    path = "customers.csv"

df = pd.read_csv(path)
print("Loaded:", path)

## 3. Dataset exploration
`head()`, `tail()`, `info()`, `shape`, `dtypes`, and missing values.

In [ ]:
print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
print("Target variable:", TARGET)
print("Feature columns:", [c for c in df.columns if c != TARGET])
print(f"Samples vs features: {df.shape[0]} vs {df.shape[1] - 1}")
print(f"Storage requirement: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
df.dtypes.to_frame("dtype")

In [ ]:
df.isnull().sum().to_frame('missing values')

## 4. EDA: data quality checks
### 4.1 Missing values and completeness

In [ ]:
miss = df.isnull().sum()
quality = pd.DataFrame({
    "missing": miss,
    "missing_%": (miss / len(df) * 100).round(2),
    "completeness_%": (100 - miss / len(df) * 100).round(2),
})
quality

In [ ]:
plt.figure(figsize=(8, 4))
quality["missing_%"].sort_values().plot.barh(color="#2f6fb5")
plt.xlabel("Missing (%)"); plt.title("Missing values per column")
plt.tight_layout(); plt.show()

### 4.2 Duplicate records

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(10)

### 4.3 Outlier detection (IQR rule and z-score)

In [ ]:
num_cols = df.select_dtypes("number").columns.drop(TARGET).tolist()
rows = {}
for c in num_cols:
    s = df[c].dropna()
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows[c] = {"lower_fence": round(lo, 2), "upper_fence": round(hi, 2),
               "IQR_outliers": int(((s < lo) | (s > hi)).sum()),
               "z>3_outliers": int((np.abs(stats.zscore(s)) > 3).sum())}
pd.DataFrame(rows).T

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(3 * len(num_cols), 4))
for ax, c in zip(axes, num_cols):
    sns.boxplot(y=df[c], ax=ax, color="#7fb3e6"); ax.set_title(c)
plt.tight_layout(); plt.show()

### 4.4 Data inconsistency validation and data types

In [ ]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
for c in cat_cols:
    vals = df[c].dropna().unique()
    clean = pd.Series(vals).str.strip().str.lower().nunique()
    print(f"{c}: {sorted(map(repr, vals))}")
    if len(vals) != clean:
        print(f"   -> {len(vals) - clean} inconsistent variant(s) (case / extra spaces)")

In [ ]:
pd.DataFrame({"dtype": df.dtypes.astype(str), "unique values": df.nunique()})

### 4.5 Label accuracy

In [ ]:
print("Valid labels:", sorted(df[TARGET].dropna().unique()))
print("Missing labels:", df[TARGET].isnull().sum())

## 5. EDA: pattern analysis

### 5.1 Class balance

In [ ]:
bal = df[TARGET].value_counts(normalize=True).mul(100).round(2)
print(bal.to_string())
print(f"Imbalance ratio: {bal.max() / bal.min():.2f} : 1")
sns.countplot(x=TARGET, data=df, hue=TARGET, palette="Blues_d", legend=False)
plt.title("Class balance"); plt.show()

### 5.2 Feature distributions

In [ ]:
n = len(num_cols); rows_ = int(np.ceil(n / 3))
fig, axes = plt.subplots(rows_, 3, figsize=(15, 4 * rows_))
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(df[c].dropna(), kde=True, ax=ax, color="#2f6fb5"); ax.set_title(c)
for ax in axes.ravel()[n:]: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 4))
for ax, c in zip(axes, cat_cols):
    tmp = df.assign(**{c: df[c].str.strip().str.title()})
    sns.barplot(data=tmp, x=c, y=TARGET, ax=ax, color="#2f6fb5")
    ax.set_title(f"Purchase rate by {c}"); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

### 5.3 Correlation matrix

In [ ]:
corr = df[num_cols + [TARGET]].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation heatmap"); plt.show()
corr[TARGET].drop(TARGET).sort_values(ascending=False).to_frame("correlation with Purchase")

### 5.4 Feature relationships

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="Time_On_Site_Min", y="Annual_Income", hue=TARGET, palette="coolwarm", alpha=.6)
plt.title("Time on site vs income"); plt.show()

In [ ]:
sns.pairplot(df[["Time_On_Site_Min", "Pages_Viewed", "Previous_Purchases", TARGET]].dropna(),
             hue=TARGET, palette="coolwarm", corner=True, plot_kws={"alpha": .5})
plt.show()

## 6. EDA: statistical insights

In [ ]:
df[num_cols].describe().T.round(2)

In [ ]:
sk = pd.DataFrame({"skew": df[num_cols].skew().round(3), "kurtosis": df[num_cols].kurt().round(3)})
sk["note"] = np.where(sk["skew"].abs() > 1, "highly skewed: consider log transform", "ok")
sk

### Hypothesis testing
For each feature we test whether buyers and non-buyers differ. Numeric features use Welch's t-test, categorical features use the chi-square test. A p-value below 0.05 means the feature is related to purchase.

In [ ]:
rows = []
for c in num_cols:
    a = df.loc[df[TARGET] == 1, c].dropna(); b = df.loc[df[TARGET] == 0, c].dropna()
    t, p = stats.ttest_ind(a, b, equal_var=False)
    rows.append((c, "t-test", round(t, 3), p))
for c in cat_cols:
    tab = pd.crosstab(df[c].str.strip().str.title(), df[TARGET])
    chi, p, *_ = stats.chi2_contingency(tab)
    rows.append((c, "chi-square", round(chi, 3), p))
ht = pd.DataFrame(rows, columns=["feature", "test", "statistic", "p_value"]).sort_values("p_value")
ht["significant (p<0.05)"] = ht["p_value"] < 0.05
ht.reset_index(drop=True)

In [ ]:
key = ht.loc[ht["significant (p<0.05)"], "feature"].tolist()
print("Key predictive features:", key)

## 7. Preprocessing decisions
| Problem | Decision | Action |
|---|---|---|
| Duplicate rows | **Remove** | `drop_duplicates()` |
| Inconsistent text (`male `) | **Transform** | strip + title-case |
| Missing numeric values | **Impute** | median |
| Missing categorical values | **Impute** | most frequent |
| Different scales / skew | **Transform** | `StandardScaler` |
| Categorical columns | **Transform** | one-hot encoding |
| Class imbalance | handle | `class_weight="balanced"` |

In [ ]:
d = df.drop_duplicates().copy()
for c in cat_cols:
    d[c] = d[c].str.strip().str.title()
d = d.dropna(subset=[TARGET])
print("Rows before:", len(df), "| after cleaning:", len(d))

X, y = d.drop(columns=[TARGET]), d[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 8. Baseline models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
}
fitted, results = {}, []
for name, m in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", m)]).fit(X_train, y_train)
    pred, proba = pipe.predict(X_test), pipe.predict_proba(X_test)[:, 1]
    fitted[name] = (pipe, pred, proba)
    results.append({"model": name, "accuracy": accuracy_score(y_test, pred),
                    "precision": precision_score(y_test, pred), "recall": recall_score(y_test, pred),
                    "f1": f1_score(y_test, pred), "roc_auc": roc_auc_score(y_test, proba)})
pd.DataFrame(results).set_index("model").round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for name, (pipe, pred, proba) in fitted.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, proba):.2f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=.4); axes[0].set_title("ROC curves")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate"); axes[0].legend()
for ax, (name, (pipe, pred, proba)) in zip(axes[1:], fitted.items()):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(f"{name}: confusion matrix"); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

### Feature importance (Random Forest)

In [ ]:
rf_pipe = fitted["Random Forest"][0]
names = rf_pipe.named_steps["prep"].get_feature_names_out()
imp = pd.Series(rf_pipe.named_steps["model"].feature_importances_, index=names).sort_values()
imp.tail(10).plot.barh(color="#2f6fb5", figsize=(8, 4.5)); plt.title("Top features"); plt.tight_layout(); plt.show()

### Predict for a new customer

In [ ]:
new_customer = pd.DataFrame([{
    "Age": 35, "Gender": "Female", "Annual_Income": 62000, "Region": "South", "Device": "Desktop",
    "Time_On_Site_Min": 14.0, "Pages_Viewed": 9, "Previous_Purchases": 3, "Email_Opened": 1}])
best = fitted["Random Forest"][0]
p = best.predict_proba(new_customer)[0, 1]
print(f"Purchase probability: {p:.1%} -> {'WILL PURCHASE' if p >= .5 else 'will not purchase'}")

## 9. Summary report
**Dataset:** see section 3 for samples, features, data types and storage.
**Quality issues found:** missing values in `Age`, `Annual_Income`, `Device`; duplicate rows; extreme income outliers; inconsistent spelling in `Gender`.
**Patterns:** moderate class imbalance; `Previous_Purchases`, `Time_On_Site_Min`, `Pages_Viewed`, `Email_Opened` and `Annual_Income` are the strongest signals; `Annual_Income` and `Time_On_Site_Min` are right-skewed.
**Decisions:** remove duplicates, fix text, impute, scale, one-hot encode, use class weights.
**Next:** hyperparameter tuning, cross-validation, XGBoost / Gradient Boosting, deployment as an API, explainable AI.

*All numbers in this notebook come from a synthetic sample dataset. Re-run on the real dataset for final results.*